[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/15-projects/p05-rag-course-chatbot/colab_quickstart.ipynb)

# P5 quickstart - RAG chatbot over the course notes

Indexes the curriculum's markdown files with sentence-transformers + FAISS, answers questions with citations, rewrites follow-ups, and evaluates retrieval. CPU is enough. First run downloads small Hugging Face models (~90 MB embedder, ~720 MB SmolLM2-360M).

## 1. Get the code
In Colab this clones the repo. If the project folder already exists, the clone is skipped.

In [1]:
import os, subprocess, sys
from pathlib import Path

PROJECT = "15-projects/p05-rag-course-chatbot"
if Path("main.py").exists() and Path("requirements.txt").exists():
    print("Already inside the project folder - no clone needed.")
elif Path("../../" + PROJECT).exists():
    os.chdir("../../" + PROJECT)          # repo already on disk two levels up
else:
    if not Path("AI-Engineer").exists():  # first run in Colab: download the repo
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/vvvvvivekkk/AI-Engineer.git"], check=True)
    os.chdir(Path("AI-Engineer") / PROJECT)
print("Working folder:", Path.cwd())

Already inside the project folder - no clone needed.
Working folder: /home/claude/AI-Engineer/15-projects/p05-rag-course-chatbot


## 2. Install packages (Colab only)

In [2]:
import sys, subprocess
from pathlib import Path
# Only install in Colab. Locally you already did "pip install -r requirements.txt" in a venv.
# Colab ships its own PyTorch, so we skip the torch line to save a big download.
if "google.colab" in sys.modules:
    reqs = [l for l in Path("requirements.txt").read_text().splitlines()
            if l.strip() and not l.startswith(("#", "torch"))]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
    print("installed", len(reqs), "packages")
else:
    print("Not in Colab - skipping install.")

Not in Colab - skipping install.


## 3. Build the index over every markdown file in the repo
Each file is cut into chunks by heading; each chunk becomes a 384-number embedding in FAISS.

In [3]:
!{sys.executable} main.py index

Loading weights: 100%|████████████████████| 103/103 [00:00<00:00, 41309.36it/s]


index: 1956 chunks from 164 markdown files (repo)


## 4. Ask questions - extractive mode (no LLM, instant)
The answer quotes the best-matching sentences and cites file > heading.

In [4]:
from src.index import get_index
from src.chat import RAGChatbot
from src import config
vi = get_index(folder=config.INDEX_DIR / 'repo')
bot = RAGChatbot(vi, answer_mode='extractive')
for q in ['What is a virtual environment?', 'Why should I use one?']:
    a = bot.ask(q)
    print('Q:', q, '| searched for:', a.standalone)
    print(a.formatted(), end='\n\n')

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 41788.87it/s]

Q: What is a virtual environment? | searched for: What is a virtual environment?
What is a virtual environment and why use one? (junior)  A virtual environment is an isolated folder with its own Python interpreter and installed packages. [2] A virtual machine (VM) pretends to be a whole separate computer with its own operating system, which is heavy. [1] We name the environment .venv (the dot makes it a hidden folder; most tools and VS Code look for this name automatically). [4]

Sources:
[1] 14-deployment-and-llmops/concepts.md > Module 14 — Deployment and LLMOps > 3. Docker from zero — the kitchen in a box > 3.2 The four words you must know
[2] 16-careers/interview-question-bank.md > Interview question bank (137 questions with model answers) > Python
[3] 00-setup/concepts.md > Module 00 — Setting up your AI engineering toolkit > 9. Virtual environments: one spice rack per project > 9.1 Create and activate
[4] 00-setup/concepts.md > Module 00 — Setting up your AI engineering toolkit >

Q: Why should I use one? | searched for: Why should I use one (virtual environment)?
use one? (junior)  A virtual environment is an isolated folder with its own Python interpreter and installed packages. [4] A virtual machine (VM) pretends to be a whole separate computer with its own operating system, which is heavy. [2] Virtual environment (venv) — a folder containing its own copy of Python's package space. [1]

Sources:
[1] 00-setup/concepts.md > Module 00 — Setting up your AI engineering toolkit > 9. Virtual environments: one spice rack per project
[2] 14-deployment-and-llmops/concepts.md > Module 14 — Deployment and LLMOps > 3. Docker from zero — the kitchen in a box > 3.2 The four words you must know
[3] 00-setup/README.md > Module 00 — Setup: tools, environment and accounts (week 1) > Which tool for this module, and why
[4] 16-careers/interview-question-bank.md > Interview question bank (137 questions with model answers) > Python



## 5. The same with a local LLM writing the answer
SmolLM2-360M runs on CPU (slow-ish, 10 s to a few minutes). Its answers read well but can add unsupported claims; the sources list is always shown so you can check. Set LLM_PROVIDER in .env to use Ollama, Claude or OpenAI instead.

In [5]:
bot_llm = RAGChatbot(vi, answer_mode='llm')
print(bot_llm.ask('What is retrieval-augmented generation?').formatted())

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:  65%|██████▍   | 188/290 [00:00<00:00, 1879.13it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1689.82it/s]

Retrieval-augmented generation (RAG) is a technique in natural language processing (NLP) that allows a model to generate answers to questions by retrieving relevant chunks of text from a large dataset. This approach enables the model to generate answers that are more accurate and relevant than those generated by a traditional retrieval system. RAG is particularly useful for tasks such as question answering, where the goal is to generate a response that is as accurate as possible given the available information. The process involves first retrieving relevant chunks of text from a dataset, then reranking them to find the most relevant ones, and finally assembling a prompt that includes the retrieved chunks and the question. This approach allows for more accurate and relevant answers than those generated by traditional retrieval systems.

Sources:
[1] 09-rag/README.md > Module 09 — Retrieval-Augmented Generation (RAG) (weeks 15–16) > Time estimate
[2] 09-rag/concepts.md > Module 09 — Retr

## 6. Evaluate: two chunk sizes x reranker off/on
Uses the committed sample docs and FAQ, so numbers are repeatable.

In [6]:
!{sys.executable} main.py eval

Loading weights: 100%|████████████████████| 103/103 [00:00<00:00, 41688.05it/s]


Loading weights: 100%|████████████████████| 105/105 [00:00<00:00, 39522.74it/s]


 chunk_words  chunks  reranker  hit@1  hit@4   MRR  answer_token_f1  paraphrase_hit@1  paraphrase_MRR
          40      53     False  0.938  1.000 0.953            0.345             0.750           0.812
          40      53      True  0.969  0.969 0.969            0.339             0.875           0.875
         180      36     False  0.875  1.000 0.927            0.336             0.500           0.708
         180      36      True  0.969  0.969 0.969            0.325             0.875           0.875

written to /home/claude/AI-Engineer/15-projects/p05-rag-course-chatbot/reports/eval.md


## 7. Run the tests

In [7]:
!{sys.executable} -m pytest -q

....

.

.

.

..

.                                                               [100%]
10 passed in 15.98s


## 8. Web UI (Colab only)
Starts Gradio with a public share link. Locally run `python app.py` instead.

In [8]:
if 'google.colab' in sys.modules:
    os.environ['GRADIO_SHARE'] = 'true'
    !{sys.executable} app.py
else:
    print('Locally: run  python app.py  and open http://127.0.0.1:7860')

Locally: run  python app.py  and open http://127.0.0.1:7860
